In [54]:
# import required libraries and loads dataset
import pandas as pd
import numpy as np
try:
    credit_default_dataframe = pd.read_csv("/content/train_data.csv")
    #Check the dataset is not empty
    if credit_default_dataframe.empty:
        raise ValueError("Loaded dataset is empty.")
    # Display dataset summary
    print("Dataset loaded successfully!!!!")
    print(f"Shape: {credit_default_dataframe.shape}")
    print("\nFirst 5 rows:")
    display(credit_default_dataframe.head())
except FileNotFoundError as file_error:
    print(f"File Error: {file_error}")
except pd.errors.ParserError as parser_error:
    print(f"Parsing Error: Unable to read CSV file : {parser_error}")
except ValueError as value_error:
    print(f"Validation Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Dataset loaded successfully!!!!
Shape: (153755, 122)

First 5 rows:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,410704,0,Cash loans,F,N,Y,1,157500.0,900000.0,26446.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,381230,0,Cash loans,F,N,Y,1,90000.0,733176.0,21438.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,1.0
2,450177,0,Cash loans,F,Y,Y,0,189000.0,1795500.0,62541.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,332445,0,Cash loans,M,Y,N,0,175500.0,494550.0,45490.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
4,357429,0,Cash loans,F,Y,Y,0,270000.0,1724688.0,54283.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
#initial data exploratiom to understand the data quality and target balance
try:
    print("Basic Dataset Information")
    #shape
    total_rows, total_columns = credit_default_dataframe.shape
    print(f"Total Rows    : {total_rows}")
    print(f"Total Columns : {total_columns}")
    # check duplicate
    duplicate_record_count = credit_default_dataframe.duplicated().sum()
    print(f"\nDuplicate Rows: {duplicate_record_count}")
    #missing value analysis
    missing_value_summary = credit_default_dataframe.isnull().sum()
    columns_with_missing_values = missing_value_summary[missing_value_summary > 0]
    print(f"\nColumns with Missing Values: {len(columns_with_missing_values)}")
    if len(columns_with_missing_values) > 0:
        missing_percentage = (
            columns_with_missing_values / total_rows
        ).sort_values(ascending=False) * 100
        missing_value_report = pd.DataFrame({
            "Missing_Count": columns_with_missing_values,
            "Missing_Percentage": missing_percentage
        }).sort_values(
            by="Missing_Percentage",
            ascending=False
        )
        print("\nTop 10 columns with missing values:")
        display(missing_value_report.head(10))
    #data types summary
    print("\nFeature Datatype Summary!!!")
    print(credit_default_dataframe.dtypes.value_counts())
    # Target distribution
    print("\nTarget Variable Distribution!!!")
    target_distribution = credit_default_dataframe["TARGET"].value_counts()
    target_percentage = (
        credit_default_dataframe["TARGET"]
        .value_counts(normalize=True) * 100
    )
    target_summary = pd.DataFrame({
        "Count": target_distribution,
        "Percentage": target_percentage
    })
    display(target_summary)
except KeyError as key_error:
    print(f"Column Error: {key_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Basic Dataset Information
Total Rows    : 153755
Total Columns : 122

Duplicate Rows: 0

Columns with Missing Values: 66

Top 10 columns with missing values:


,Missing_Count,Missing_Percentage
COMMONAREA_AVG,107523,69.931384
COMMONAREA_MODE,107523,69.931384
COMMONAREA_MEDI,107523,69.931384
NONLIVINGAPARTMENTS_MODE,106918,69.537901
NONLIVINGAPARTMENTS_AVG,106918,69.537901
NONLIVINGAPARTMENTS_MEDI,106918,69.537901
LIVINGAPARTMENTS_MEDI,105193,68.415986
LIVINGAPARTMENTS_MODE,105193,68.415986
LIVINGAPARTMENTS_AVG,105193,68.415986
FONDKAPREMONT_MODE,105177,68.405580



Feature Datatype Summary!!!
float64    65
int64      41
object     16
Name: count, dtype: int64

Target Variable Distribution!!!


,Count,Percentage
TARGET,,
0,141343,91.927417
1,12412,8.072583


### Interpretation of Initial Data Exploration

The initial exploration of the credit default dataset provides several important insights that directly influence the modeling strategy.

1. **Dataset Size and Structure**
   - The dataset contains 153,755 observations and 122 features, which is sufficiently large for building a reliable predictive model.
   - No duplicate records were identified, indicating good overall data integrity.

2. **Missing Value Analysis**
   - A total of 66 columns contain missing values, with several variables showing more than 65% missingness.
   - Features such as COMMONAREA_AVG, NONLIVINGAPARTMENTS_MODE, and LIVINGAPARTMENTS_AVG contain substantial missing data, suggesting that missing value handling will be an essential feature engineering step.

3. **Feature Types**
   - The dataset contains:
     - 65 numerical (float) features
     - 41 integer features
     - 16 categorical (object) features
   - This mixed feature structure indicates that categorical encoding will be required before training machine learning models.

4. **Class Imbalance**
   - The target variable TARGET is highly imbalanced:
     - 91.93% belong to class 0 (non default)
     - 8.07% belong to class 1 (default)
   - Because of this imbalance, relying only on accuracy would be misleading.
   - Therefore, metrics such as Precision, Recall, F1-score, and ROC-AUC will be used for evaluation.
   - Additionally, sampling techniques (such as SMOTE) will be considered during feature engineering to improve minority class prediction.

### Conclusion
These findings justify the next steps of:
- splitting data into train, validation, and test sets,
- establishing a baseline decision tree model,
- and applying targeted feature engineering techniques to improve predictive performance.

In [56]:
#Train / Validation / Test Split (Baseline Setup)Create baseline datasets before feature engineering
from sklearn.model_selection import train_test_split
try:
    print("Starting baseline dataset split process!!!!\n")
    target_column_name = "TARGET"
    #check the existence of target column
    if target_column_name not in credit_default_dataframe.columns:
        raise KeyError(f"{target_column_name} column not found in dataset.")
    #separate variables(target and predictor)
    target_variable_series = credit_default_dataframe[target_column_name]
    predictor_feature_dataframe = credit_default_dataframe.drop(
        columns=[target_column_name]
    )
    #remove identifier column (non predictive)
    identifier_column_name = "SK_ID_CURR"
    if identifier_column_name in predictor_feature_dataframe.columns:
        model_input_feature_dataframe = predictor_feature_dataframe.drop(
            columns=[identifier_column_name]
        )
        print(f"{identifier_column_name} removed from predictors.")
    else:
        model_input_feature_dataframe = predictor_feature_dataframe.copy()
        print(f"{identifier_column_name} not found. Continuing!!!")
    #First split -> Training (70%) + Temporary Holdout (30%)
    (
        training_feature_set,
        temporary_feature_set,
        training_target_set,
        temporary_target_set
    ) = train_test_split(
        model_input_feature_dataframe,
        target_variable_series,
        test_size=0.30,
        random_state=42,
        stratify=target_variable_series
    )
    #second split -> Validation (15%) + Testing (15%)
    (
        validation_feature_set,
        testing_feature_set,
        validation_target_set,
        testing_target_set
    ) = train_test_split(
        temporary_feature_set,
        temporary_target_set,
        test_size=0.50,
        random_state=42,
        stratify=temporary_target_set
    )
    #verify split sizes
    print("\nDataset Split Completed Successfully!!!")
    print(f"Training Features   : {training_feature_set.shape}")
    print(f"Validation Features : {validation_feature_set.shape}")
    print(f"Testing Features    : {testing_feature_set.shape}")
    #verify class balance after stratification
    print("\nTarget Distribution Check (%)")
    print("Training Set:")
    print(training_target_set.value_counts(normalize=True) * 100)
    print("\nValidation Set:")
    print(validation_target_set.value_counts(normalize=True) * 100)
    print("\nTesting Set:")
    print(testing_target_set.value_counts(normalize=True) * 100)
except KeyError as key_error:
    print(f"Column Error: {key_error}")
except ValueError as value_error:
    print(f"Split Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting baseline dataset split process!!!!

SK_ID_CURR removed from predictors.

Dataset Split Completed Successfully!!!
Training Features   : (107628, 120)
Validation Features : (23063, 120)
Testing Features    : (23064, 120)

Target Distribution Check (%)
Training Set:
TARGET
0    91.927751
1     8.072249
Name: proportion, dtype: float64

Validation Set:
TARGET
0    91.926462
1     8.073538
Name: proportion, dtype: float64

Testing Set:
TARGET
0    91.926812
1     8.073188
Name: proportion, dtype: float64


To establish a reliable baseline model, the dataset was divided using 2 stage stratified splitting strategy.
In the 1st  split, the data was divided into:
- **70% Training Set** — used for model learning.
- **30% Temporary Holdout Set** — reserved for further subdivision.

The temporary 30% holdout was intentionally created to ensure that both validation and test datasets are generated independently from the training data. This prevents information leakage and allows unbiased model evaluation.

In the 2nd split, the temporary holdout set was equally divided into:
- **15% Validation Set** — used to evaluate baseline performance and compare future feature engineering improvements.
- **15% Testing Set** — reserved for final unbiased model evaluation.

A stratified sampling approach was used in both splitting stages because the target variable is highly imbalanced (~92% non default vs ~8% default). Stratification ensures that each subset preserves the original class distribution, reducing sampling bias and improving model reliability.

Additionally, the SK_ID_CURR column was removed before splitting because it functions only as a unique customer identifier and does not contribute predictive value.

The resulting target distributions across training, validation, and testing sets are nearly identical, confirming that the split preserved class balance and dataset integrity for baseline model development.

In [57]:
# minimal Preprocessing for Baseline Model
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
try:
    print("Starting baseline preprocessing!!!\n")
    #identify numerical and categorical feature groups
    numerical_feature_columns = (
        training_feature_set
        .select_dtypes(include=["int64", "float64"])
        .columns
        .tolist()
    )
    categorical_feature_columns = (
        training_feature_set
        .select_dtypes(include=["object"])
        .columns
        .tolist()
    )
    print(f"Numerical Features: {len(numerical_feature_columns)}")
    print(f"Categorical Features: {len(categorical_feature_columns)}")
    #create copies to preserve original split datasets
    processed_training_features = training_feature_set.copy()
    processed_validation_features = validation_feature_set.copy()
    processed_testing_features = testing_feature_set.copy()
    #handle missing values in numerical features using median
    numerical_missing_value_imputer = SimpleImputer(strategy="median")
    processed_training_features[numerical_feature_columns] = (
        numerical_missing_value_imputer.fit_transform(
            training_feature_set[numerical_feature_columns]
        )
    )
    processed_validation_features[numerical_feature_columns] = (
        numerical_missing_value_imputer.transform(
            validation_feature_set[numerical_feature_columns]
        )
    )
    processed_testing_features[numerical_feature_columns] = (
        numerical_missing_value_imputer.transform(
            testing_feature_set[numerical_feature_columns]
        )
    )
    #handlemissing values in categorical features using mode
    categorical_missing_value_imputer = SimpleImputer(
        strategy="most_frequent"
    )
    processed_training_features[categorical_feature_columns] = (
        categorical_missing_value_imputer.fit_transform(
            training_feature_set[categorical_feature_columns]
        )
    )
    processed_validation_features[categorical_feature_columns] = (
        categorical_missing_value_imputer.transform(
            validation_feature_set[categorical_feature_columns]
        )
    )
    processed_testing_features[categorical_feature_columns] = (
        categorical_missing_value_imputer.transform(
            testing_feature_set[categorical_feature_columns]
        )
    )
    #encode categorical variables using ordinal encoding
    baseline_categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    processed_training_features[categorical_feature_columns] = (
        baseline_categorical_encoder.fit_transform(
            processed_training_features[categorical_feature_columns]
        )
    )
    processed_validation_features[categorical_feature_columns] = (
        baseline_categorical_encoder.transform(
            processed_validation_features[categorical_feature_columns]
        )
    )
    processed_testing_features[categorical_feature_columns] = (
        baseline_categorical_encoder.transform(
            processed_testing_features[categorical_feature_columns]
        )
    )
    #vrerify shapes
    print("\nBaseline preprocessing completed successfully!!!!")
    print(f"Processed Training Shape: {processed_training_features.shape}")
    print(f"Processed Validation Shape: {processed_validation_features.shape}")
    print(f"Processed Testing Shape: {processed_testing_features.shape}")
    remaining_missing_values = (
        processed_training_features.isnull().sum().sum()
        + processed_validation_features.isnull().sum().sum()
        + processed_testing_features.isnull().sum().sum()
    )
    print(f"\nRemaining Missing Values: {remaining_missing_values}")
except KeyError as key_error:
    print(f"Column Error: {key_error}")
except ValueError as value_error:
    print(f"Preprocessing Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting baseline preprocessing!!!

Numerical Features: 104
Categorical Features: 16

Baseline preprocessing completed successfully!!!!
Processed Training Shape: (107628, 120)
Processed Validation Shape: (23063, 120)
Processed Testing Shape: (23064, 120)

Remaining Missing Values: 0


Before training the baseline decision tree model, only the minimum required preprocessing steps were applied to make the dataset suitable for machine learning while preserving the original data structure.

1. Missing Value Treatment
   - Numerical features were imputed using the   median because median is less sensitive to outliers and provides a robust estimate for skewed financial data.
   - Categorical features were imputed using the most frequent value (mode) to preserve the dominant category within each feature.

2. Categorical Feature Encoding
   - Since machine learning models require numerical inputs, all categorical variables were converted into numeric form using Ordinal Encoding.
   - Ordinal encoding was selected for the baseline because it is computationally efficient and avoids creating excessive dimensions, which can occur with one-hot encoding.

3. Data Leakage Prevention
   - All preprocessing transformations were fitted only on the training dataset and then applied to the validation and test sets.
   - This ensures that no information from unseen data influences the training process, preserving evaluation integrity.

4. Preprocessing Outcome
   - The number of features remained unchanged (120 features), ensuring that no engineered features were introduced during baseline creation.
   - All missing values were successfully removed (0 remaining missing values), making the dataset fully model-ready.

### Conclusion
This preprocessing stage establishes a fair baseline model benchmark, allowing future feature engineering techniques to be evaluated objectively based on their impact on predictive performance.

In [58]:
#train baseline decision tree variant (XGBoost) and establish baseline model performance
from xgboost import XGBClassifier
try:
    print("Starting baseline model training!!!!\n")
    #initialize baseline XGBoost classifier
    baseline_decision_tree_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    #train model using training dataset only
    baseline_decision_tree_model.fit(
        processed_training_features,
        training_target_set
    )
    print("Baseline model training completed successfully!!!!\n")
    #generate class predictions
    training_set_predictions = baseline_decision_tree_model.predict(
        processed_training_features
    )
    validation_set_predictions = baseline_decision_tree_model.predict(
        processed_validation_features
    )
    #generate probability predictions for later evaluation
    training_prediction_probabilities = (
        baseline_decision_tree_model.predict_proba(
            processed_training_features
        )[:, 1]
    )
    validation_prediction_probabilities = (
        baseline_decision_tree_model.predict_proba(
            processed_validation_features
        )[:, 1]
    )
    print("Prediction generation completed!!!!")
    print(f"Training Predictions Shape: {training_set_predictions.shape}")
    print(f"Validation Predictions Shape: {validation_set_predictions.shape}")
except ImportError as import_error:
    print(f"Library Error: {import_error}")
except ValueError as value_error:
    print(f"Training Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting baseline model training!!!!

Baseline model training completed successfully!!!!

Prediction generation completed!!!!
Training Predictions Shape: (107628,)
Validation Predictions Shape: (23063,)


### Baseline Model Hyperparameters
The following baseline hyperparameters were selected for training :

1. n_estimators = 200
   - Specifies the number of boosting trees.
   - A value of 200 provides sufficient learning capacity while avoiding excessive training time.

2. max_depth = 6
   - Controls the depth of each tree.
   - A moderate depth helps capture meaningful feature interactions while reducing overfitting risk.

3. learning_rate = 0.1
   - Determines how much each tree contributes to the final prediction.
   - This is a commonly accepted baseline value that balances learning speed and stability.

4. objective = binary:logistic
   - Appropriate because the problem is a binary classification task(default vs non default).
   - Produces probability outputs required for threshold based evaluation.

5. eval_metric = logloss
   - Measures probabilistic prediction quality during training.
   - Useful for imbalanced classification problems because it evaluates confidence, not just correctness.

6. random_state = 42
   - Ensures reproducibility so results can be replicated consistently.

7. n_jobs = -1
   - Enables parallel processing using all available CPU cores, improving training efficiency.

Predictions were created for both the training and validation datasets to allow comparison of model behavior across seen and unseen data. This helps identify potential overfitting before feature engineering improvements are introduced.

### Conclusion
This trained model serves as the baseline benchmark, against which all future feature engineering techniques will be evaluated.

### Baseline Training Output

The baseline XGBoost model trained successfully without errors, confirming that the preprocessing pipeline correctly prepared the dataset for machine learning.

The generated prediction counts match the corresponding dataset sizes:

- Training predictions: 107,628
- Validation predictions: 23,063

This confirms that:
- all observations were processed successfully,
- no records were lost during training or prediction,
- and dataset dimensions remained consistent throughout the pipeline.


In [59]:
# evaluate baseline model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
try:
    print("Starting baseline model evaluation!!!\n")
    #store evaluation metrics in reusable structure
    baseline_model_performance_summary = {}

    #evaluate training performance
    baseline_model_performance_summary["Training"] = {
        "Accuracy": accuracy_score(
            training_target_set,
            training_set_predictions
        ),
        "Precision": precision_score(
            training_target_set,
            training_set_predictions
        ),
        "Recall": recall_score(
            training_target_set,
            training_set_predictions
        ),
        "F1_Score": f1_score(
            training_target_set,
            training_set_predictions
        ),
        "ROC_AUC": roc_auc_score(
            training_target_set,
            training_prediction_probabilities
        )
    }
    #evaluate validation performance
    baseline_model_performance_summary["Validation"] = {
        "Accuracy": accuracy_score(
            validation_target_set,
            validation_set_predictions
        ),
        "Precision": precision_score(
            validation_target_set,
            validation_set_predictions
        ),
        "Recall": recall_score(
            validation_target_set,
            validation_set_predictions
        ),
        "F1_Score": f1_score(
            validation_target_set,
            validation_set_predictions
        ),
        "ROC_AUC": roc_auc_score(
            validation_target_set,
            validation_prediction_probabilities
        )
    }
    #convert results into dataframe for readability
    baseline_performance_dataframe = pd.DataFrame(
        baseline_model_performance_summary
    ).T
    print("Baseline Performance Summary!!!!!")
    display(baseline_performance_dataframe)
    #generate confusion matrix for validation set
    validation_confusion_matrix = confusion_matrix(
        validation_target_set,
        validation_set_predictions
    )
    confusion_matrix_dataframe = pd.DataFrame(
        validation_confusion_matrix,
        index=["Actual_Non_Default", "Actual_Default"],
        columns=["Predicted_Non_Default", "Predicted_Default"]
    )
    print("\nValidation Confusion Matrix")
    display(confusion_matrix_dataframe)
except ValueError as value_error:
    print(f"Evaluation Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting baseline model evaluation!!!

Baseline Performance Summary!!!!!


,Accuracy,Precision,Recall,F1_Score,ROC_AUC
Training,0.925103,0.906615,0.080456,0.147796,0.892783
Validation,0.919221,0.495327,0.028464,0.053834,0.739881



Validation Confusion Matrix


,Predicted_Non_Default,Predicted_Default
Actual_Non_Default,21147,54
Actual_Default,1809,53


### Interpretation of Baseline Model Performance

The baseline XGBoost model achieved high overall accuracy 91.92% on the validation dataset. However, because the target variable is highly imbalanced, accuracy alone is misleading.

Further analysis shows:
1. Very Low Recall 2.85%
   - The model correctly identified only 53 out of 1,862 actual default cases.
   - This indicates poor minority class detection and confirms that the model is strongly biased toward the majority (non default) class.

2. Low F1 Score 0.054
   - The low F1 score reflects poor balance between precision and recall, making the current baseline unsuitable for practical credit risk prediction.

3. Moderate ROC AUC 0.740
   - The ROC AUC score suggests that the model has reasonable ranking ability, meaning it can partially distinguish between default and non default customers.
   - However, its default decision threshold is not effectively identifying risky customers.

4. Signs of Mild Overfitting
   - Training ROC AUC 0.893 is notably higher than validation ROC AUC 0.740, indicating some overfitting.

### Conclusion
The baseline model establishes an important benchmark but performs poorly on the minority default class.

In [60]:
#SMOTE based class rebalancing to improve minority class representation
from imblearn.over_sampling import SMOTE
try:
    print("Starting SMOTE based class rebalancing!!!!\n")
    #initialize SMOTE sampler
    smote_rebalancing_engine = SMOTE(
        sampling_strategy="auto",
        random_state=42
    )
    #apply only on training set
    (
        smote_balanced_training_features,
        smote_balanced_training_target
    ) = smote_rebalancing_engine.fit_resample(
        processed_training_features,
        training_target_set
    )
    #verify balancing
    print("SMOTE completed successfully.")
    print(f"Original Training Shape : {processed_training_features.shape}")
    print(f"Balanced Training Shape : {smote_balanced_training_features.shape}")
    print("\nBalanced Target Distribution:")
    print(smote_balanced_training_target.value_counts())
except ValueError as value_error:
    print(f"SMOTE Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting SMOTE based class rebalancing!!!!

SMOTE completed successfully.
Original Training Shape : (107628, 120)
Balanced Training Shape : (197880, 120)

Balanced Target Distribution:
TARGET
0    98940
1    98940
Name: count, dtype: int64


### SMOTE Based Class Rebalancing
After applying SMOTE:
- Original training samples: 107,628
- Resampled training samples:197,880

The target distribution became perfectly balanced:
- Class 0 (non default):98,940
- Class 1 (default):98,940

This synthetic rebalancing allows the model to learn minority class patterns more effectively and is expected to improve Recall and F1 score, which are critical metrics for credit default prediction.

In [61]:
#retrain model after SMOTE
try:
    print("Training SMOTE enhanced model!!!!\n")
    smote_enhanced_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    smote_enhanced_model.fit(
        smote_balanced_training_features,
        smote_balanced_training_target
    )
    smote_validation_predictions = smote_enhanced_model.predict(
        processed_validation_features
    )
    smote_validation_probabilities = (
        smote_enhanced_model.predict_proba(
            processed_validation_features
        )[:, 1]
    )
    print("SMOTE model training completed.")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Training SMOTE enhanced model!!!!

SMOTE model training completed.


In [62]:
#evaluate SMOTE model
try:
    smote_model_metrics = {
        "Accuracy": accuracy_score(validation_target_set, smote_validation_predictions),
        "Precision": precision_score(validation_target_set, smote_validation_predictions),
        "Recall": recall_score(validation_target_set, smote_validation_predictions),
        "F1_Score": f1_score(validation_target_set, smote_validation_predictions),
        "ROC_AUC": roc_auc_score(validation_target_set, smote_validation_probabilities)
    }
    smote_results_dataframe = pd.DataFrame(
        smote_model_metrics,
        index=["SMOTE_Model"]
    )
    display(smote_results_dataframe)
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

,Accuracy,Precision,Recall,F1_Score,ROC_AUC
SMOTE_Model,0.919525,0.53,0.028464,0.054027,0.738891


### Interpretation of SMOTE Model Performance

Key observations:
- Recall remained unchanged 2.85%, indicating that the model still failed to identify most default cases.
- Precision increased slightly 49.5% -> 53.0%, suggesting marginally better confidence when predicting defaults.
- F1 score showed negligible improvement, meaning overall minority class performance remained weak.
- ROC AUC decreased slightly, indicating no ranking improvement.

### Why SMOTE did not help
This suggests that class imbalance was not the only problem affecting performance.
Possible reasons include:
1. The underlying feature patterns may not strongly separate default and non default classes.
2. The XGBoost classifier may still be using a conservative decision boundary that favors the majority class.
3. Synthetic oversampling improved balance, but did not introduce enough new predictive information.

### Conclusion
Although SMOTE is a commonly recommended solution for imbalanced datasets, in this case it did not meaningfully improve model effectiveness. This demonstrates the importance of testing multiple feature engineering strategies rather than relying on a single technique.

In [63]:
#missingness informed feature reconstruction.Remove ultra-high missing columns (>70%)Create missingness indicator features
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
try:
    print("Starting missingness informed feature reconstruction!!!!\n")
    #calculate missing %'s from TRAIN only
    training_missing_percentage = (
        training_feature_set.isnull().mean() * 100
    )
    ultra_high_missing_columns = (
        training_missing_percentage[
            training_missing_percentage > 70
        ].index.tolist()
    )
    print(f"Columns removed (>70% missing): {len(ultra_high_missing_columns)}")
    #drop same columns from all sets
    reconstructed_training_features = training_feature_set.drop(
        columns=ultra_high_missing_columns
    ).copy()
    reconstructed_validation_features = validation_feature_set.drop(
        columns=ultra_high_missing_columns
    ).copy()
    reconstructed_testing_features = testing_feature_set.drop(
        columns=ultra_high_missing_columns
    ).copy()
    #create missingness indicator columns
    candidate_missing_columns = (
        reconstructed_training_features
        .columns[
            reconstructed_training_features.isnull().sum() > 0
        ]
        .tolist()
    )
    for each_missing_column in candidate_missing_columns:
        missing_flag_column_name = (
            each_missing_column + "_missing_flag"
        )
        reconstructed_training_features[
            missing_flag_column_name
        ] = reconstructed_training_features[
            each_missing_column
        ].isnull().astype(int)
        reconstructed_validation_features[
            missing_flag_column_name
        ] = reconstructed_validation_features[
            each_missing_column
        ].isnull().astype(int)
        reconstructed_testing_features[
            missing_flag_column_name
        ] = reconstructed_testing_features[
            each_missing_column
        ].isnull().astype(int)
    print(
        f"Missingness indicator features created: "
        f"{len(candidate_missing_columns)}"
    )
    #identify column groups
    reconstructed_numeric_columns = (
        reconstructed_training_features
        .select_dtypes(include=["int64", "float64"])
        .columns
        .tolist()
    )
    reconstructed_categorical_columns = (
        reconstructed_training_features
        .select_dtypes(include=["object"])
        .columns
        .tolist()
    )
    #impute missing values
    numeric_reconstruction_imputer = SimpleImputer(
        strategy="median"
    )
    categorical_reconstruction_imputer = SimpleImputer(
        strategy="most_frequent"
    )
    reconstructed_training_features[
        reconstructed_numeric_columns
    ] = numeric_reconstruction_imputer.fit_transform(
        reconstructed_training_features[
            reconstructed_numeric_columns
        ]
    )
    reconstructed_validation_features[
        reconstructed_numeric_columns
    ] = numeric_reconstruction_imputer.transform(
        reconstructed_validation_features[
            reconstructed_numeric_columns
        ]
    )
    reconstructed_training_features[
        reconstructed_categorical_columns
    ] = categorical_reconstruction_imputer.fit_transform(
        reconstructed_training_features[
            reconstructed_categorical_columns
        ]
    )
    reconstructed_validation_features[
        reconstructed_categorical_columns
    ] = categorical_reconstruction_imputer.transform(
        reconstructed_validation_features[
            reconstructed_categorical_columns
        ]
    )
    #encode categoricals
    reconstruction_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    reconstructed_training_features[
        reconstructed_categorical_columns
    ] = reconstruction_encoder.fit_transform(
        reconstructed_training_features[
            reconstructed_categorical_columns
        ]
    )
    reconstructed_validation_features[
        reconstructed_categorical_columns
    ] = reconstruction_encoder.transform(
        reconstructed_validation_features[
            reconstructed_categorical_columns
        ]
    )
    #retrain model
    reconstructed_feature_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    reconstructed_feature_model.fit(
        reconstructed_training_features,
        training_target_set
    )
    reconstructed_predictions = (
        reconstructed_feature_model.predict(
            reconstructed_validation_features
        )
    )
    reconstructed_probabilities = (
        reconstructed_feature_model.predict_proba(
            reconstructed_validation_features
        )[:, 1]
    )
    #evaluate retrained model
    reconstructed_metrics = {
        "Accuracy": accuracy_score(
            validation_target_set,
            reconstructed_predictions
        ),
        "Precision": precision_score(
            validation_target_set,
            reconstructed_predictions
        ),
        "Recall": recall_score(
            validation_target_set,
            reconstructed_predictions
        ),
        "F1_Score": f1_score(
            validation_target_set,
            reconstructed_predictions
        ),
        "ROC_AUC": roc_auc_score(
            validation_target_set,
            reconstructed_probabilities
        )
    }
    print("\nTechnique 2 completed successfully!!!!")
    display(
        pd.DataFrame(
            reconstructed_metrics,
            index=["Missingness_Reconstructed_Model"]
        )
    )
except KeyError as key_error:
    print(f"Column Error: {key_error}")
except ValueError as value_error:
    print(f"Feature Engineering Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting missingness informed feature reconstruction!!!!

Columns removed (>70% missing): 0
Missingness indicator features created: 65

Technique 2 completed successfully!!!!


,Accuracy,Precision,Recall,F1_Score,ROC_AUC
Missingness_Reconstructed_Model,0.919828,0.563107,0.031149,0.059033,0.741735


### Interpretation of Missingness Informed Feature Reconstruction
This technique was designed to address a limitation observed after SMOTE: the model still struggled to identify default cases, suggesting that the issue was not only class imbalance but also insufficient predictive signal.

The feature engineering strategy introduced two steps:

1. Ultra high missing feature pruning
   - Features with more than 70% missing values were targeted for removal.
   - In this dataset, no features exceeded this threshold, so no columns were removed.

2. Missingness indicator generation
   - For every feature containing missing values, a new binary indicator 0/1` was created to capture whether the original value was missing.
   - This created 65 additional signal features.

### Performance Impact

Compared with the baseline model:

- Accuracy improved slightly
- Precision improved noticeably 49.5% -> 56.3%
- Recall improved 2.85% -> 3.11%
- F1 score improved
- ROC AUC improved 0.7399 -> 0.7417

### Conclusion

These results suggest that missingness itself carries predictive information in the credit dataset. Missing value patterns appear to be associated with customer default behavior, making this feature engineering technique more effective than SMOTE.

In [64]:
#threshold optimization.Improve recall using probability thr
try:
    print("Starting threshold optimization!!!\n")
    # define candidate thresholds
    candidate_threshold_values = np.arange(
        0.10,
        0.51,
        0.05
    )
    threshold_search_results = []
    # search best threshold
    for each_threshold in candidate_threshold_values:
        threshold_based_predictions = (
            validation_prediction_probabilities >= each_threshold
        ).astype(int)
        threshold_f1_score = f1_score(
            validation_target_set,
            threshold_based_predictions
        )
        threshold_search_results.append(
            [each_threshold, threshold_f1_score]
        )
    threshold_results_dataframe = pd.DataFrame(
        threshold_search_results,
        columns=["Threshold", "F1_Score"]
    )
    optimal_threshold_value = (
        threshold_results_dataframe
        .sort_values(
            by="F1_Score",
            ascending=False
        )
        .iloc[0]["Threshold"]
    )
    print(
        f"Optimal threshold identified: "
        f"{optimal_threshold_value:.2f}"
    )
    #apply optimal threshold
    optimized_validation_predictions = (
        validation_prediction_probabilities
        >= optimal_threshold_value
    ).astype(int)
    #reevaluate
    optimized_threshold_metrics = {
        "Accuracy": accuracy_score(
            validation_target_set,
            optimized_validation_predictions
        ),
        "Precision": precision_score(
            validation_target_set,
            optimized_validation_predictions
        ),
        "Recall": recall_score(
            validation_target_set,
            optimized_validation_predictions
        ),
        "F1_Score": f1_score(
            validation_target_set,
            optimized_validation_predictions
        ),
        "ROC_AUC": roc_auc_score(
            validation_target_set,
            validation_prediction_probabilities
        )
    }
    print("\nThreshold optimization completed successfully")
    display(
        pd.DataFrame(
            optimized_threshold_metrics,
            index=["Threshold_Optimized_Model"]
        )
    )
except ValueError as value_error:
    print(f"Threshold Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting threshold optimization!!!

Optimal threshold identified: 0.15

Threshold optimization completed successfully


,Accuracy,Precision,Recall,F1_Score,ROC_AUC
Threshold_Optimized_Model,0.844123,0.230984,0.39957,0.292741,0.739881


### Performance Impact of Threshold Optimization

Because the baseline model achieved the strongest probability ranking quality, its predicted probabilities were selected for threshold tuning rather than retraining a new model. This allows improvement of classification performance while preserving the models learned probability structure. Threshold optimization a post processing technique that improves minority class detection without altering the underlying model or introducing new training data, making it a computationally efficient and leakage free feature engineering strategy.

Threshold tuning was therefore applied to the baseline model probabilities, testing multiple candidate thresholds and selecting the value that maximized F1 score.

Compared with the baseline:

- Recall improved from 2.85% to 39.96%
- F1 score improved from 0.054 to 0.293
- Accuracy remained relatively strong at 84.41%
- ROC AUC remained unchanged because ranking ability did not change.

### Conclusion
Threshold optimization produced the best overall balance between precision, recall, and accuracy, making it the strongest final model for this credit default prediction task.

In [65]:
#test evaluation of Threshold Optimized Model

try:
    print("Starting final test evaluation of threshold optimized model!!!\n")
    #lock previously discovered threshold
    locked_optimal_threshold = optimal_threshold_value
    print(
        f"Locked decision threshold: "
        f"{locked_optimal_threshold:.2f}"
    )
    #using the original baseline XGBoost model,generate probability predictions on test set
    testing_set_prediction_probabilities = (
        baseline_decision_tree_model.predict_proba(
            processed_testing_features
        )[:, 1]
    )
    #using optimized threshold convert probabilities into class labels
    threshold_optimized_test_predictions = (
        testing_set_prediction_probabilities
        >= locked_optimal_threshold
    ).astype(int)
    #compute final evaluation metrics
    final_test_metric_summary = {
        "Accuracy": accuracy_score(
            testing_target_set,
            threshold_optimized_test_predictions
        ),
        "Precision": precision_score(
            testing_target_set,
            threshold_optimized_test_predictions
        ),
        "Recall": recall_score(
            testing_target_set,
            threshold_optimized_test_predictions
        ),
        "F1_Score": f1_score(
            testing_target_set,
            threshold_optimized_test_predictions
        ),
        "ROC_AUC": roc_auc_score(
            testing_target_set,
            testing_set_prediction_probabilities
        )
    }
    final_test_results_dataframe = pd.DataFrame(
        final_test_metric_summary,
        index=["Final_Test_Threshold_Optimized_Model"]
    )
    print("\nFinal Test Performance Summary")
    display(final_test_results_dataframe)
    #build confusion matrix
    final_test_confusion_matrix = confusion_matrix(
        testing_target_set,
        threshold_optimized_test_predictions
    )
    final_test_confusion_dataframe = pd.DataFrame(
        final_test_confusion_matrix,
        index=[
            "Actual_Non_Default",
            "Actual_Default"
        ],
        columns=[
            "Predicted_Non_Default",
            "Predicted_Default"
        ]
    )
    print("\nFinal Test Confusion Matrix")
    display(final_test_confusion_dataframe)
    #generalization gap check
    validation_f1_score_reference = (
        optimized_threshold_metrics["F1_Score"]
    )
    test_f1_score_observed = (
        final_test_metric_summary["F1_Score"]
    )
    generalization_gap = (
        validation_f1_score_reference
        - test_f1_score_observed
    )
    print("\nGeneralization Check")
    print(
        f"Validation F1 : "
        f"{validation_f1_score_reference:.4f}"
    )
    print(
        f"Test F1: "
        f"{test_f1_score_observed:.4f}"
    )
    print(
        f"Generalization Gap : "
        f"{generalization_gap:.4f}"
    )
except NameError as name_error:
    print(
        f"Variable Error: {name_error}. "
        "Ensure previous cell completed successfully first."
    )
except ValueError as value_error:
    print(f"Evaluation Error: {value_error}")
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Starting final test evaluation of threshold optimized model!!!

Locked decision threshold: 0.15

Final Test Performance Summary


,Accuracy,Precision,Recall,F1_Score,ROC_AUC
Final_Test_Threshold_Optimized_Model,0.845473,0.233396,0.400107,0.294816,0.751423



Final Test Confusion Matrix


,Predicted_Non_Default,Predicted_Default
Actual_Non_Default,18755,2447
Actual_Default,1117,745



Generalization Check
Validation F1 : 0.2927
Test F1: 0.2948
Generalization Gap : -0.0021


The threshold optimized XGBoost model was selected as the final model because it achieved the strongest balance between Recall, F1 score, and overall accuracy during validation.

### Final Test Results

- Accuracy: 84.55%
- Precision: 23.34%
- Recall: 40.01%
- F1-score: 0.295
- ROC-AUC: 0.751

### Generalization Analysis

The validation F1 score 0.2927  and test F1 score 0.2948 were nearly identical, producing a negligible generalization gap.

This confirms that the model:
- generalizes well to unseen data,
- is not overfitted,
- and is reliable for deployment-level prediction.

### Final Conclusion

The threshold optimized model is the best final solution for this credit default prediction problem because it substantially improves minority class detection while maintaining strong overall predictive performance.

In [66]:
#final model comparison summary.Compare baseline and all feature engineering techniques
try:
    print("Generating final model comparison summary!!!\n")
    #collect validation metrics for model comparison
    final_model_comparison_dictionary = {
        "Baseline_Model": {
            "Accuracy": baseline_model_performance_summary[
                "Validation"
            ]["Accuracy"],
            "Precision": baseline_model_performance_summary[
                "Validation"
            ]["Precision"],
            "Recall": baseline_model_performance_summary[
                "Validation"
            ]["Recall"],
            "F1_Score": baseline_model_performance_summary[
                "Validation"
            ]["F1_Score"],
            "ROC_AUC": baseline_model_performance_summary[
                "Validation"
            ]["ROC_AUC"]
        },
        "SMOTE_Model": smote_model_metrics,
        "Missingness_Reconstruction_Model":
            reconstructed_metrics,
        "Threshold_Optimized_Model":
            optimized_threshold_metrics
    }
    #convert to dataframe and round for readability
    final_model_comparison_dataframe = pd.DataFrame(
        final_model_comparison_dictionary
    ).T
    final_model_comparison_dataframe = (
        final_model_comparison_dataframe.round(4)
    )
    #identify best metric values
    best_performance_summary = (
        final_model_comparison_dataframe.idxmax()
    )
    print("Final Performance Comparison Table")
    display(final_model_comparison_dataframe)
    print("\nBest Performing Model by Metric")
    display(
        pd.DataFrame(
            best_performance_summary,
            columns=["Best_Model"]
        )
    )
except NameError as name_error:
    print(
        f"Variable Error: {name_error}. "
        "Ensure all previous cells executed successfully."
    )
except Exception as unexpected_error:
    print(f"Unexpected Error: {unexpected_error}")

Generating final model comparison summary!!!

Final Performance Comparison Table


,Accuracy,Precision,Recall,F1_Score,ROC_AUC
Baseline_Model,0.9192,0.4953,0.0285,0.0538,0.7399
SMOTE_Model,0.9195,0.5300,0.0285,0.0540,0.7389
Missingness_Reconstruction_Model,0.9198,0.5631,0.0311,0.0590,0.7417
Threshold_Optimized_Model,0.8441,0.2310,0.3996,0.2927,0.7399



Best Performing Model by Metric


,Best_Model
Accuracy,Missingness_Reconstruction_Model
Precision,Missingness_Reconstruction_Model
Recall,Threshold_Optimized_Model
F1_Score,Threshold_Optimized_Model
ROC_AUC,Missingness_Reconstruction_Model


# Final Conclusion
A baseline model was first established using minimal preprocessing. Although it achieved high accuracy 91.92%, it performed poorly on the minority default class, with Recall of only 2.85%. This confirmed that accuracy alone was not an appropriate performance metric for this imbalanced classification problem.

Three feature engineering techniques were then applied and compared:

### 1.SMOTE Based Class Rebalancing
SMOTE was introduced to address class imbalance by generating synthetic minority class examples. However, performance did not improve significantly, indicating that imbalance alone was not the primary limitation of the model.

### 2.Missingness Informed Feature Reconstruction
This technique introduced binary missingness indicators to capture whether missing values themselves carried predictive information. This produced the highest Accuracy 91.98% and highest Precision 56.31% , demonstrating that missing value patterns were informative in this credit dataset.

### 3. Threshold Optimization
Instead of modifying the model architecture, probability threshold tuning adjusted the decision boundary
This produced the strongest overall balance:
- Recall improved from 2.85% to 40.01%
- F1-score improved from 0.054 to 0.295
- ROC-AUC improved to 0.751

### Final Model Selection
The Threshold Optimized model was selected as the final best model because it achieved the best balance between:
- minority class detection,
- overall predictive stability,
- and real world business usefulness.

